# 01 — Data Collection

Fetches market data from World Bank API, ITU, and BEREC.
Output: `data/raw/world_bank.csv` and panel data structures.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))

from src.data.fetcher import load_config, fetch_all
from src.data.preprocessor import align_panel, impute_missing, add_pstn_phaseout_feature, add_derived_features, save_processed
import pandas as pd
pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 120)

In [ ]:
config = load_config("config.yaml")
print("Config loaded.")
print(f"Countries: {sum(len(v) for v in config['countries'].values())} total")
print(f"World Bank indicators: {list(config['data']['world_bank']['indicators'].keys())}")

In [ ]:
all_data = fetch_all(config)
wb = all_data['world_bank']
berec_dates = all_data['berec_dates']
print(f"World Bank data: {wb.shape[0]} rows, {wb.shape[1]} cols")
print(f"BEREC dates extracted: {len(berec_dates)} countries")

In [ ]:
if wb.empty:
    print("World Bank API unavailable or empty. Using synthetic fallback data.")
    # Generate realistic synthetic data for demonstration
    countries_list = []
    for region in config['countries'].values():
        for item in region:
            for code, name in item.items():
                countries_list.append((code, name))
    np.random.seed(42)
    rows = []
    for code, name in countries_list:
        for year in range(2000, 2026):
            base = 50.0 if code not in ('in', 'cn', 'br') else 5.0
            decay = 1.0 / (1.0 + np.exp(-0.25 * (year - 2005)))
            val = base * (1.0 - decay * 0.9) + np.random.normal(0, 1.5)
            broadband = min(45, (year - 2000) * 2.0 + np.random.normal(0, 2))
            gdp = 30000 + (year - 2000) * 500 + np.random.normal(0, 1000)
            urban = 65 + (year - 2000) * 0.3 + np.random.normal(0, 1)
            rows.append({
                'country': code,
                'year': year,
                'fixed_subs_value': max(0, val),
                'broadband_value': broadband,
                'gdp_per_capita_value': gdp,
                'urban_pop_value': urban,
            })
    wb = pd.DataFrame(rows)
    print(f"Generated synthetic panel: {wb.shape[0]} rows")

print(wb.head())

In [ ]:
panel = align_panel(
    {'world_bank': wb},
    countries=wb['country'].unique().tolist(),
    year_range=range(2000, 2026),
)
print(f"Panel before imputation: {panel.shape}")
panel = impute_missing(panel, method='linear', max_gap=3)
panel = add_pstn_phaseout_feature(panel, config['data']['pstn_switchoff'])
panel = add_derived_features(panel)
print(f"Panel after processing: {panel.shape}")
print(f"Columns: {list(panel.columns)}")

In [ ]:
save_processed(panel)
print("Panel saved to data/processed/panel_data.csv")

---
### Summary

In [ ]:
print(f"\nCountries in panel: {panel['country'].nunique()}")
print(f"Year range: {panel['year'].min()} - {panel['year'].max()}")
print(f"\nPSTN phaseout status:\n{panel.groupby('country')['has_pstn_phaseout'].max().value_counts().to_string()}")
print(f"\nMissing values:\n{panel.isnull().sum()[panel.isnull().sum() > 0].to_string()}")